# Energy Consumption Variance Decomposition

In [17]:
import duckdb
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import f_oneway
from pprint import pprint

## Global Variables

In [3]:
DATABASE_PATH = "../../database/eBuS.duckdb"
VEHICLE_DICT = {"Ebusco2.2electric12m": "Ebusco 2.2", "SolarisUrbino12electric": "Urbino 12", "SolarisUrbino18electric": "Urbino 18"}

In [52]:
con = duckdb.connect(DATABASE_PATH)

df = con.execute("""
    SELECT 
        tripinfo_id,
        battery_totalEnergyConsumed,
        battery_totalEnergyRegenerated, 
        tripinfo_routeLength,
        tripinfo_vType,
        scenario,
        seed
    FROM multirun_tripinfo
    WHERE scenario IN ('summer', 'winter', 'reduced', 'increased')
""").fetchdf()

con.close()


# Energy consumed per distance
df["energy_efficiency"] = (
    (df["battery_totalEnergyConsumed"] - df["battery_totalEnergyRegenerated"])
    / df["tripinfo_routeLength"]
)

for scenario in [
    "summer", 
    "winter", 
    ]:

    mean_instance_collection = {}
    std_instance_collection = {}
    count_instance_collection = {}

    print("\n" + "#" * 80)
    print(f"SCENARIO: {scenario.upper()}")
    print("#" * 80)

    for seed in [65, 67, 68, 70, 75]:

        subset = df[
            (df["scenario"] == scenario) &
            (df["seed"] == seed)
        ].copy()

        vtypes = sorted(subset["tripinfo_vType"].dropna().unique())

        print("\n" + "=" * 70)
        print(f"Energy efficiency statistics — {scenario.upper()} — Seed {seed}")
        print("=" * 70)

        for vtype in vtypes:

            values = subset.loc[
                subset["tripinfo_vType"] == vtype,
                "energy_efficiency"
            ].dropna()

            vehicle_name = VEHICLE_DICT.get(vtype, vtype)

            print(f"\nVehicle type: {vehicle_name}")
            print("-" * 40)
            print(values.describe().to_string())

            mean_instance_collection.setdefault(vtype, []).append(values.mean())
            std_instance_collection.setdefault(vtype, []).append(values.std())
            count_instance_collection.setdefault(vtype, []).append(values.count())

    # ---------------------------------------------------------
    # Mean-of-means across seeds (unchanged)
    # ---------------------------------------------------------
    print("\n" + "#" * 80)
    print(f"MEAN INSTANCE STATISTICS — {scenario.upper()}")
    print("#" * 80)

    for vtype, means in mean_instance_collection.items():
        vehicle_name = VEHICLE_DICT.get(vtype, vtype)
        means = pd.Series(means)
        print(f"\nVehicle type: {vehicle_name}")
        print("-" * 40)
        print(means.describe().to_string())

    # ---------------------------------------------------------
    # Average standard deviation per vehicle type, across seeds
    # ---------------------------------------------------------
    print("\n" + "#" * 80)
    print(f"AVERAGE STD ACROSS SEEDS — {scenario.upper()}")
    print("#" * 80)

    for vtype, stds in std_instance_collection.items():
        vehicle_name = VEHICLE_DICT.get(vtype, vtype)
        stds = pd.Series(stds)
        print(f"\nVehicle type: {vehicle_name}")
        print("-" * 40)
        print(f"Per-seed stds: {stds.tolist()}")
        print(f"Average std across seeds: {stds.mean():.6f}")

    # ---------------------------------------------------------
    # Conditional variance decomposition across seeds
    # Var(Y) = E[Var(Y|Seed)] + Var(E[Y|Seed])
    #            "within"           "between"
    # ---------------------------------------------------------
    print("\n" + "#" * 80)
    print(f"VARIANCE DECOMPOSITION (Seed) — {scenario.upper()}")
    print("#" * 80)

    for vtype in mean_instance_collection:

        vehicle_name = VEHICLE_DICT.get(vtype, vtype)

        n_i = pd.Series(count_instance_collection[vtype]).to_numpy(dtype=float)
        mean_i = pd.Series(mean_instance_collection[vtype]).to_numpy(dtype=float)
        std_i = pd.Series(std_instance_collection[vtype]).to_numpy(dtype=float)
        var_i = std_i ** 2

        N = n_i.sum()
        grand_mean = (n_i * mean_i).sum() / N

        # E[Var(Y|Seed)]: weighted average of per-seed variances (within)
        within = (n_i * var_i).sum() / N

        # Var(E[Y|Seed]): weighted variance of per-seed means (between)
        between = (n_i * (mean_i - grand_mean) ** 2).sum() / N

        total = within + between
        icc = between / total if total > 0 else float("nan")

        print(f"\nVehicle type: {vehicle_name}")
        print("-" * 40)
        print(f"N (total across seeds)        = {int(N)}")
        print(f"Within  E[Var(Y|Seed)]        = {within:.8f}  (SD = {within ** 0.5:.6f})")
        print(f"Between Var(E[Y|Seed])        = {between:.8e}  (SD = {between ** 0.5:.6f})")
        print(f"Total   Var(Y)                = {total:.8f}  (SD = {total ** 0.5:.6f})")
        print(f"Anteil Between / ICC           = {icc * 100:.4f} %")


################################################################################
SCENARIO: SUMMER
################################################################################

Energy efficiency statistics — SUMMER — Seed 65

Vehicle type: Ebusco 2.2
----------------------------------------
count    71.000000
mean      0.774599
std       0.028889
min       0.697941
25%       0.757633
50%       0.778368
75%       0.796594
max       0.819912

Vehicle type: Urbino 12
----------------------------------------
count    54.000000
mean      0.780090
std       0.030339
min       0.721275
25%       0.759952
50%       0.786916
75%       0.803758
max       0.819042

Vehicle type: Urbino 18
----------------------------------------
count    269.000000
mean       1.682983
std        0.049825
min        1.452721
25%        1.655709
50%        1.680585
75%        1.718720
max        1.803605

Energy efficiency statistics — SUMMER — Seed 67

Vehicle type: Ebusco 2.2
---------------------------------

### Anova Creation

Nominal Variable: Seed  
Metric Variable: Effizienz

In [56]:
con = duckdb.connect(DATABASE_PATH)

df = con.execute("""
    SELECT 
        tripinfo_id,
        battery_totalEnergyConsumed,
        battery_totalEnergyRegenerated, 
        tripinfo_routeLength,
        tripinfo_vType,
        scenario,
        seed
    FROM multirun_tripinfo
    WHERE scenario IN ('summer', 'winter', 'reduced', 'increased')
""").fetchdf()

con.close()


# Energy consumed per distance
df["energy_efficiency"] = (
    (df["battery_totalEnergyConsumed"] - df["battery_totalEnergyRegenerated"])
    / df["tripinfo_routeLength"]
)

seeds = [65, 67, 68, 70, 75]

energy_efficiency = {}

for scenario in ['summer', 'winter','reduced', 'increased']:

    energy_efficiency[scenario] = {}

    vtypes = sorted(
        df.loc[
            df["scenario"] == scenario,
            "tripinfo_vType"
        ].dropna().unique()
    )

    for vtype in vtypes:

        energy_efficiency[scenario][vtype] = {}

        for seed in seeds:

            values = df.loc[
                (df["scenario"] == scenario) &
                (df["tripinfo_vType"] == vtype) &
                (df["seed"] == seed),
                "energy_efficiency"
            ].dropna().tolist()

            energy_efficiency[scenario][vtype][seed] = values

for scenario, vtypes in energy_efficiency.items():

    for vtype, seeds in vtypes.items():

        groups = [
            values
            for values in seeds.values()
            if len(values) > 0
        ]

        if len(groups) >= 2:
            f_statistic, p_value = f_oneway(*groups)

            print(f"{scenario} - {vtype}")
            print(f"F-statistic: {f_statistic}")
            print(f"P-value: {p_value}")

summer - Ebusco2.2electric12m
F-statistic: 0.000858380585380422
P-value: 0.9999985196592587
summer - SolarisUrbino12electric
F-statistic: 0.001003757997987136
P-value: 0.9999979724878505
summer - SolarisUrbino18electric
F-statistic: 0.008177838129389802
P-value: 0.9998675022704826
winter - Ebusco2.2electric12m
F-statistic: 0.0009586725691130897
P-value: 0.9999981537785532
winter - SolarisUrbino12electric
F-statistic: 0.00346589784435859
P-value: 0.9999759070331959
winter - SolarisUrbino18electric
F-statistic: 0.00941483154688735
P-value: 0.9998246768902962
reduced - Ebusco2.2electric12m
F-statistic: 0.0008854526574283597
P-value: 0.999998424868715
reduced - SolarisUrbino12electric
F-statistic: 0.0010949547793462585
P-value: 0.9999975876284413
reduced - SolarisUrbino18electric
F-statistic: 0.0081778381293898
P-value: 0.9998675022704826
increased - Ebusco2.2electric12m
F-statistic: 0.0008583805853804221
P-value: 0.9999985196592587
increased - SolarisUrbino12electric
F-statistic: 0.001003

### Redone Variance Decomposition
Var(Y) = E + Var(E)

E stellt die durchschnittliche Varianz von Y innerhalb jeder durch X definierten Gruppe dar (interne Variabilität).  
Var(E) ist die Varianz der bedingten Mittelwerte von Y gemäß X (Variabilität zwischen Gruppen).


In [57]:
con = duckdb.connect(DATABASE_PATH)

df = con.execute("""
    SELECT 
        tripinfo_id,
        battery_totalEnergyConsumed,
        battery_totalEnergyRegenerated, 
        tripinfo_routeLength,
        tripinfo_vType,
        scenario,
        seed
    FROM multirun_tripinfo
    WHERE scenario IN ('summer', 'winter', 'reduced', 'increased')
""").fetchdf()

con.close()


# Energy consumed per distance
df["energy_efficiency"] = (
    (df["battery_totalEnergyConsumed"] - df["battery_totalEnergyRegenerated"])
    / df["tripinfo_routeLength"]
)

seeds = [65, 67, 68, 70, 75]

energy_efficiency = {}

for scenario in ['summer', 'winter','reduced', 'increased']:

    energy_efficiency[scenario] = {}

    vtypes = sorted(
        df.loc[
            df["scenario"] == scenario,
            "tripinfo_vType"
        ].dropna().unique()
    )

    for vtype in vtypes:

        energy_efficiency[scenario][vtype] = {}

        for seed in seeds:

            values = df.loc[
                (df["scenario"] == scenario) &
                (df["tripinfo_vType"] == vtype) &
                (df["seed"] == seed),
                "energy_efficiency"
            ].dropna().tolist()

            energy_efficiency[scenario][vtype][seed] = values



group_statistics = {}

for scenario, vtypes in energy_efficiency.items():

    group_statistics[scenario] = {}

    for vtype, seeds in vtypes.items():

        group_statistics[scenario][vtype] = {}

        for seed, values in seeds.items():

            values = np.asarray(values, dtype=float)

            if len(values) == 0:
                continue

            group_statistics[scenario][vtype][seed] = {
                "n": len(values),
                "mean": np.mean(values),
                "std": np.std(values),
                "variance": np.var(values)
            }

pprint(group_statistics)

for scenario, vtypes in group_statistics.items():

    for vtype, seeds in vtypes.items():

        variances = [group["variance"] for group in seeds.values()]
        means = [group["mean"] for group in seeds.values()]

        e = np.mean(variances)
        var_e = np.var(means)

        print(scenario, vtype)
        print(f"Var(Y) = {e:.8f} + {var_e:.8f}")
        print(f"       = {e + var_e:.8f}")

{'increased': {'Ebusco2.2electric12m': {65: {'mean': np.float64(0.7745994507136563),
                                             'n': 71,
                                             'std': np.float64(0.028684451069983005),
                                             'variance': np.float64(0.0008227977331862492)},
                                        67: {'mean': np.float64(0.7745430884348643),
                                             'n': 71,
                                             'std': np.float64(0.02845373341661851),
                                             'variance': np.float64(0.0008096149453439928)},
                                        68: {'mean': np.float64(0.7746591629039722),
                                             'n': 71,
                                             'std': np.float64(0.02862928303027473),
                                             'variance': np.float64(0.0008196358468275766)},
                                        70: {'me

In [54]:
# AI-generated on 2026-09-26

import duckdb
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Daten laden
# ------------------------------------------------------------

con = duckdb.connect(DATABASE_PATH)

df = con.execute("""
    SELECT 
        tripinfo_id,
        battery_totalEnergyConsumed,
        battery_totalEnergyRegenerated,
        tripinfo_routeLength,
        tripinfo_vType,
        scenario,
        seed
    FROM multirun_tripinfo
    WHERE scenario IN ('summer', 'winter')
""").fetchdf()

con.close()


# ------------------------------------------------------------
# 2. Energy Efficiency berechnen
# ------------------------------------------------------------

df["energy_efficiency"] = (
    (
        df["battery_totalEnergyConsumed"]
        - df["battery_totalEnergyRegenerated"]
    )
    / df["tripinfo_routeLength"]
)


# Nur gültige Beobachtungen verwenden
df = df.dropna(
    subset=[
        "energy_efficiency",
        "scenario",
        "tripinfo_vType",
        "seed"
    ]
)


# ------------------------------------------------------------
# 3. Varianzdekomposition
# ------------------------------------------------------------

results = []

for (scenario, vtype), group in df.groupby(
    ["scenario", "tripinfo_vType"]
):

    # Gesamtvarianz über alle Trips
    total_variance = group["energy_efficiency"].var(ddof=0)

    # Gesamtmittelwert
    overall_mean = group["energy_efficiency"].mean()

    # Statistiken pro Seed
    seed_stats = (
        group
        .groupby("seed")["energy_efficiency"]
        .agg(
            n="count",
            mean="mean",
            variance=lambda x: x.var(ddof=0)
        )
    )

    # Gewichte entsprechend der Anzahl der Trips
    weights = seed_stats["n"] / seed_stats["n"].sum()


    # --------------------------------------------------------
    # Within-Seed-Varianz
    #
    # E[Var(Y | Seed)]
    # --------------------------------------------------------

    within_variance = np.sum(
        weights * seed_stats["variance"]
    )


    # --------------------------------------------------------
    # Between-Seed-Varianz
    #
    # Var(E[Y | Seed])
    # --------------------------------------------------------

    between_variance = np.sum(
        weights
        * (seed_stats["mean"] - overall_mean) ** 2
    )


    # --------------------------------------------------------
    # Kontrollwert
    #
    # Within + Between sollte der Gesamtvarianz entsprechen
    # --------------------------------------------------------

    decomposed_variance = (
        within_variance
        + between_variance
    )


    # --------------------------------------------------------
    # Prozentuale Anteile
    # --------------------------------------------------------

    within_percent = (
        within_variance / total_variance * 100
        if total_variance > 0
        else np.nan
    )

    between_percent = (
        between_variance / total_variance * 100
        if total_variance > 0
        else np.nan
    )


    # --------------------------------------------------------
    # Ergebnis speichern
    # --------------------------------------------------------

    results.append({
        "scenario": scenario,
        "vtype": vtype,
        "n_total": len(group),
        "n_seeds": group["seed"].nunique(),
        "overall_mean": overall_mean,

        "total_variance": total_variance,

        "within_variance": within_variance,
        "within_percent": within_percent,

        "between_variance": between_variance,
        "between_percent": between_percent,

        "decomposed_variance": decomposed_variance,

        "difference": (
            total_variance
            - decomposed_variance
        )
    })


# ------------------------------------------------------------
# 4. Ergebnisse als DataFrame
# ------------------------------------------------------------

variance_decomposition = pd.DataFrame(results)


# Nach Scenario und VType sortieren
variance_decomposition = variance_decomposition.sort_values(
    ["scenario", "vtype"]
)


# Ausgabe
print(
    variance_decomposition.to_string(
        index=False,
        float_format=lambda x: f"{x:.8f}"
    )
)

scenario                   vtype  n_total  n_seeds  overall_mean  total_variance  within_variance  within_percent  between_variance  between_percent  decomposed_variance  difference
  summer    Ebusco2.2electric12m      355        5    0.77462408      0.00082037       0.00082036     99.99901900        0.00000001       0.00098100           0.00082037 -0.00000000
  summer SolarisUrbino12electric      270        5    0.77989851      0.00090695       0.00090694     99.99848492        0.00000001       0.00151508           0.00090695  0.00000000
  summer SolarisUrbino18electric     1345        5    1.68309326      0.00246199       0.00246193     99.99755891        0.00000006       0.00244109           0.00246199 -0.00000000
  winter    Ebusco2.2electric12m      355        5    0.91373797      0.00101047       0.00101046     99.99890439        0.00000001       0.00109561           0.00101047  0.00000000
  winter SolarisUrbino12electric      270        5    0.92181808      0.00117644       0.0